This notebook allows to search OpenAlex using a specific search phrase, and embed and store the fecthed articles into a chroma database.

# Fetch articles from OpenAlex

##### Functions for OpenAlex search

In [ ]:
import requests

# function to parse the text
def reconstruct_text(inverted_index):
    word_index = []
    for k,v in inverted_index.items():
        for index in v:
            word_index.append([k,index])

    word_index = sorted(word_index,key = lambda x : x[1])

    word_list = []
    for i in range(len(word_index)):
        word_list.append(word_index[i][0])

    separator = ' '
    reconstructed_text = separator.join(word_list)

    return reconstructed_text

# function that uses openalex to search web for papers
def search_openalex(search_phrase, max_results=100000):
    base_url = "https://api.openalex.org/works"  # Replace with the actual API endpoint

    filters = [  
        f"default.search:{search_phrase}",
        "has_abstract:true",
        "language:en"
    ]

    # Construct the query parameters
    params = {
    "filter": str.join(",", filters),  # Only return works with abstracts
    "sort": "relevance_score:desc",
    "per_page": 200,  # maximum allowed per page ,
    "cursor": "*", 
    }

    #fetch articles from all pages
    all_results = []
    abstract_list = []

    i = 0
    total_results = 0
    while total_results < max_results and params["cursor"]:
        print("page " + str(i), "total results: " + str(total_results))
        r = requests.get(base_url, params=params)
        r.raise_for_status()
        res_json = r.json()
        
        # Add the current batch of results
        all_results.extend(res_json.get("results", []))
        total_results += len(res_json.get("results", []))
        # Update the cursor to fetch the next page
        params["cursor"] = res_json.get("meta", {}).get("next_cursor")
        i = i+1
        
        for j in range(len(res_json["results"])):
            abstract_list.append(reconstruct_text(res_json["results"][j]['abstract_inverted_index']))

    return all_results, abstract_list

In [ ]:
import time
import pandas as pd
def fetch_papers_by_ids(paper_ids, batch_size=100):
    # ------------------------------------------------------------------ #
    # 1. Sanitise the input so we always have a clean list[str]          #
    # ------------------------------------------------------------------ #
    if isinstance(paper_ids, pd.Series):
        paper_ids = paper_ids.astype(str).tolist()
    else:
        paper_ids = [str(x) for x in paper_ids]      # works for lists, arrays, etc.

    base_url = "https://api.openalex.org/works"
    results, abstract_list = [], []
    total_processed = 0

    # ------------------------------------------------------------------ #
    # 2. Process the IDs in batches                                      #
    # ------------------------------------------------------------------ #
    for i in range(0, len(paper_ids), batch_size):
        batch = paper_ids[i : i + batch_size]        # positional slice now safe
        print(f"Processing batch {i // batch_size + 1} "
              f"(IDs {i}–{i + len(batch) - 1})")

        params = {
            "filter": f"openalex_id:{'|'.join(batch)}",
            "per_page": len(batch),
            "cursor": "*"
        }

        # 3. Cursor pagination loop
        while params["cursor"]:
            try:
                response = requests.get(base_url, params=params, timeout=30)
                response.raise_for_status()
                data = response.json()

                # store full result objects
                batch_results = data.get("results", [])
                results.extend(batch_results)

                # reconstruct each abstract
                for r in batch_results:
                    if r.get("abstract_inverted_index"):
                        abstract_list.append(
                            reconstruct_text(r.get("abstract_inverted_index", {}))
                        )
                    else:
                        abstract_list.append(
                            None
                        )

                total_processed += len(batch_results)
                params["cursor"] = data.get("meta", {}).get("next_cursor")
            
            except Exception as e:
                print(f"Error on batch {i}: {e}")
                time.sleep(1)

    print(f"\nFetched {total_processed} papers out of {len(paper_ids)} requested")
    return results, abstract_list

##### OpenAlex Search

In [ ]:
search_phrase = '("cancer" OR "carcinoma" OR "tumor" OR "tumour" OR "neoplasm" OR "oncology") AND ("clinical study" OR "clinical trial" OR "randomized controlled trial" OR "randomised controlled trial" OR "RCT" OR "systematic review" OR "meta-analysis" OR "phase I trial" OR "phase II trial" OR "phase III trial" OR "controlled trial" OR "multicenter trial" OR "multicentre trial")'

In [ ]:
res_, abstract = search_openalex(search_phrase, max_results=100000)
print(f"Total results fetched: {len(res_)}")

In [ ]:
## ! ONLY RUN IF YOU WANT TO SAVE THE RESULTS retrieved from OpenAlex! ***
import pickle
with open("../saved_variables/res_.pkl", "wb") as f:
    pickle.dump(res_, f)
with open("../saved_variables/abstract.pkl", "wb") as f:
    pickle.dump(abstract, f)

##### Get Varianscape Papers From CSV List

In [ ]:
import pandas as pd
variantscape_paper_ids = pd.read_csv("../cleaned_df_v4_corrected.csv")


In [ ]:
res_variantscape, abstracts_variantscape = fetch_papers_by_ids(paper_ids = variantscape_paper_ids["PaperId"], batch_size=100)

#### Merge papers (Variantscape and OpenAlex search)

In [ ]:
total_res = res_variantscape + res_
total_abstracts = abstracts_variantscape + abstract

# Embed end store articles in chromaDB

In [ ]:
!pip install chromadb
!pip install sentence_transformers

In [ ]:
#create or get collection

import chromadb
from chromadb.utils import embedding_functions

CHROMA_DATA_PATH = "chroma_data_20250603/"
EMBED_MODEL =  "all-MiniLM-L6-v2"
COLLECTION_NAME = "searchable_db_collection_fd"

client = chromadb.PersistentClient(path=CHROMA_DATA_PATH)
embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)

existing_collections = client.list_collections()
print(existing_collections)
if COLLECTION_NAME not in [col.name for col in existing_collections]:
    collection = client.create_collection(
                                        name=COLLECTION_NAME,
                                        embedding_function=embedding_func,
                                        metadata={"hnsw:space": "cosine"})
else:
    collection = client.get_collection(name="searchable_db_collection_fd")

In [ ]:
#get titles and metadata, create IDs
string_ids = [str(i) for i in range(len(total_res))]  # Generating unique string IDs
titles = [item["title"] or "Unknown Title" for item in total_res]
metadatas = []
#Print All available metadata 
for item in total_res:
    # print(item["id"])
    # print(item.keys())
    # print(item["title"])
    # print(item["authorships"])
    # print(item["publication_year"])
    # print(item["primary_location"])
    # print(item["primary_location"]["source"])
    # print(item["primary_location"]["source"]["display_name"])
    country=""
    if item["authorships"]:
        if len(item["authorships"])>0:
            if len(item["authorships"][0]["countries"]) > 0:
               country=item["authorships"][0]["countries"][0]
               print(country)
    metadatas.append({
        "titles": item["title"] or "Unknown Title",
        "first_author": (
            item.get("authorships", [{}])[0].get("author", {}).get("display_name")
            if item.get("authorships") and len(item.get("authorships")) > 0
            else "Unknown Author"  # Fallback to "Unknown Author" if authorships is None or empty
        ),  # Fallback to "Unknown Author" if display_name is None
        "journal": (
            item.get("primary_location", {})
                .get("source", {})
                .get("display_name")
            if item.get("primary_location") and item.get("primary_location").get("source")
            else "Unknown Journal"
        ),   # Fallback to "Unknown Journal" if display_name is None
        "year": item["publication_year"] or "Unknown Publication Year",
        "openAlex_id": item["id"] or "Unknown OpenAlex ID",
        "countryMainAuthor": country
    #  "bestOAUrl": item["best_oa_location"]["landing_page_url"]   
    })




In [ ]:
print(metadatas[5])

In [ ]:
from tqdm import tqdm  # Import the progress bar library

# Define batch size (tuning this can improve performance)
batch_size = 10  # Adjust batch size as needed

# Get the total number of documents
total_docs = len(total_res)

# Loop through the data in chunks and add to the collection
for i in tqdm(range(0, total_docs, batch_size), desc="Adding data to collection", unit="batch"):
    # Create a chunk of data
    start_idx = i
    end_idx = min(i + batch_size, total_docs)
    
    documents_chunk = [
        titles[j] + " " + (total_abstracts[j] if total_abstracts[j] is not None else "")
        for j in range(start_idx, end_idx)
    ]
    ids_chunk = string_ids[start_idx:end_idx]
    metadatas_chunk = metadatas[start_idx:end_idx]
    
    # Add the current chunk to the collection
    collection.add(
        documents=documents_chunk,
        ids=ids_chunk,
        metadatas=metadatas_chunk
    )


In [ ]:
# In make_chromadb.ipynb, after the tqdm loop for collection.add:
print("\n--- Verification Step ---")
if collection.count() > 0:
    if not string_ids: # Make sure string_ids is populated from your data loading
        print("  ERROR: string_ids list is empty. Cannot perform verification.")
    else:
        test_id_to_check = string_ids[0] 
        try:
            retrieved_item_notebook = collection.get(ids=[test_id_to_check], include=['embeddings', 'documents'])
            print(f"Retrieved item '{test_id_to_check}' from notebook context:")
            
            # Check documents
            doc_text_list = retrieved_item_notebook.get('documents')
            if doc_text_list and len(doc_text_list) > 0:
                doc_text = doc_text_list[0]
                print(f"  Document (first 100 chars): {doc_text[:100]}...")
            else:
                print(f"  Document for '{test_id_to_check}' is MISSING or EMPTY.")

            # Check embeddings
            embeddings_result = retrieved_item_notebook.get('embeddings') # This could be a list of embeddings or None
            
            actual_embedding_vector = None
            
            # Determine if embeddings_result is a list and extract the first embedding
            if isinstance(embeddings_result, list):
                if len(embeddings_result) > 0:
                    actual_embedding_vector = embeddings_result[0]
                else:
                    print(f"  ERROR: Embedding list for '{test_id_to_check}' is EMPTY (list of len 0).")
            elif hasattr(embeddings_result, 'shape'): # Check if it's a NumPy array directly (less common for .get() but possible)
                # This case might indicate an unexpected return structure or a single embedding returned not in a list
                print(f"  INFO: Embeddings result appears to be a direct NumPy array. Shape: {embeddings_result.shape}")
                if embeddings_result.ndim == 1: # A single flat vector
                     actual_embedding_vector = embeddings_result
                elif embeddings_result.ndim > 1 and embeddings_result.shape[0] == 1: # An array of arrays, take the first row
                     actual_embedding_vector = embeddings_result[0]
                else:
                    print(f"  ERROR: Embeddings result is a NumPy array with unexpected shape: {embeddings_result.shape}")
            
            if actual_embedding_vector is not None:
                embedding_length = 0
                # Check if it's a numpy array by checking for 'size' attribute (that isn't a method)
                is_numpy_array = hasattr(actual_embedding_vector, 'size') and not callable(getattr(actual_embedding_vector, 'size', None))
                is_list = isinstance(actual_embedding_vector, list)

                if is_numpy_array:
                    embedding_length = actual_embedding_vector.size 
                elif is_list: # Should be a flat list of numbers if it's an embedding
                    embedding_length = len(actual_embedding_vector)
                
                if embedding_length > 0:
                    print(f"  Embedding for '{test_id_to_check}' successfully generated. Type: {type(actual_embedding_vector)}, Length/Size: {embedding_length}")
                    try:
                        sample = actual_embedding_vector[:5] # Works for lists and numpy arrays
                        print(f"  Embedding sample (first 5 dims): {sample}")
                    except Exception as e_sample:
                        print(f"  Embedding sample could not be retrieved/sliced (type: {type(actual_embedding_vector)}): {e_sample}")
                else: # length is 0
                    print(f"  ERROR: Embedding for '{test_id_to_check}' is EMPTY (length/size 0). Type: {type(actual_embedding_vector)}")
            elif isinstance(embeddings_result, list) and not embeddings_result: 
                pass # Already handled: print(f"  ERROR: Embedding list for '{test_id_to_check}' is EMPTY (list of len 0).")
            else: # actual_embedding_vector remained None and wasn't an empty list
                print(f"  ERROR: Embedding for '{test_id_to_check}' is MISSING or in an unexpected format. Embeddings result from DB: {embeddings_result}")
        
        except Exception as e:
            print(f"  ERROR during verification for item '{test_id_to_check}': {e}")
            import traceback
            traceback.print_exc()
else:
    print("Collection is empty after population attempt. No items to verify.")
print("--- End Verification Step ---\n")